# Notebook 04: Embedding EDA


Self-contained exploratory analysis of embedding geometry for both ESM-2 and AbLang2.
Covers sequence-level and residue-level delta embeddings.
All cached tensors are loaded from Drive -- no model inference in this notebook.

Analyses:
- Discriminability gain from delta computation (CoV raw vs delta)
- Delta norm distributions by dataset and by CDR/FR region
- Spearman(delta norm, DMS score) per dataset for both models and both levels
- PCA structure of delta embeddings (chain identity, region, dataset)
- Cross-model comparison: do ESM-2 and AbLang2 agree on mutation magnitude?
- Summary Spearman table: both models x both levels x 5 datasets


## Setup


In [ ]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")


Detects whether the notebook is running on Colab or locally. On Colab, mounts Drive
and clones (or pulls) the repo. Locally, resolves the repo root from the notebook's
location in `notebooks/`.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.


In [ ]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:    {DRIVE_ROOT}")
print(f"Embedding dir: {EMBEDDING_DIR}")
print(f"Figures dir:   {FIGURES_DIR}")
print("Paths set.")


`src/config.py` resolves `DRIVE_ROOT` automatically: Colab mount path first, then
Google Drive Desktop glob (any account), then `outputs/` fallback. No per-collaborator
edits needed. The figures dir is where all plots from this notebook are saved.

Expected output: paths ending in `DL_Final_Project/Antibody_Project/...` (Drive) or
`outputs/...` (local fallback).


In [ ]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")


In [ ]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")


In [ ]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")


Device check. This notebook does not load any models or run GPU inference --
all computations (norm, PCA, Spearman) run on CPU. The device check is included
for consistency with other notebooks.


## Imports


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.decomposition import PCA

from src.config import DATA_DIR, EMBEDDING_DIR, FIGURES_DIR
from src.visualization.plots import (
    plot_delta_norm_by_dataset,
    plot_delta_norm_cdr_vs_fr,
    plot_delta_pca,
    plot_delta_variance_ratio,
    plot_model_comparison_norms,
)

print("Imports OK.")


## Load Data


In [ ]:
# Load mutation metadata: region labels, DMS scores, dataset names, chain assignments
df = pd.read_csv(DATA_DIR / 'abagym_antibody.csv', dtype={'site': str})

regions   = df['region'].tolist()                          # 5318 strings: CDR_H1, ..., FR
dms_scores = df['MinMax_normalized_DMS_score'].values      # (5318,) float
dms_names  = df['DMS_name'].tolist()                       # 5318 strings: dataset per row
chains     = df['chains'].tolist()                         # 5318 strings: 'H' or 'L'

# Binary CDR/FR label (collapses all 6 CDR subtypes)
cdr_fr = ['FR' if r == 'FR' else 'CDR' for r in regions]

datasets = sorted(df['DMS_name'].unique())
dms_arr  = np.array(dms_names)

print(f"Rows: {len(df)}")
print(f"Datasets: {datasets}")
print(f"Region counts:")
print(df['region'].value_counts().to_string())


Loads `abagym_antibody.csv` for the per-mutation metadata needed throughout the EDA.
The `region` column was pre-computed during NB01 via ANARCI IMGT mapping and identifies
which CDR loop or framework region each mutation falls in.

Binary `cdr_fr` collapses CDR_H1 through CDR_L3 into a single 'CDR' label for the
two-group CDR vs FR comparisons.

Expected: 5318 rows, 7 region types, 5 datasets.


In [ ]:
# Load all cached delta tensors and raw embeddings (CPU only -- no model inference)
esm2_seq_raw    = torch.load(EMBEDDING_DIR / 'esm2_abagym.pt',                  map_location='cpu')
esm2_seq_delta  = torch.load(EMBEDDING_DIR / 'esm2_abagym_delta.pt',            map_location='cpu')
esm2_res_delta  = torch.load(EMBEDDING_DIR / 'esm2_abagym_residue_delta.pt',    map_location='cpu')

abl_seq_raw     = torch.load(EMBEDDING_DIR / 'ablang2_abagym.pt',               map_location='cpu')
abl_seq_delta   = torch.load(EMBEDDING_DIR / 'ablang2_abagym_delta.pt',         map_location='cpu')
abl_res_delta   = torch.load(EMBEDDING_DIR / 'ablang2_abagym_residue_delta.pt', map_location='cpu')

print(f"ESM-2  raw sequence:      {esm2_seq_raw.shape}")
print(f"ESM-2  sequence delta:    {esm2_seq_delta.shape}")
print(f"ESM-2  residue delta:     {esm2_res_delta.shape}")
print(f"AbLang2 raw sequence:     {abl_seq_raw.shape}")
print(f"AbLang2 sequence delta:   {abl_seq_delta.shape}")
print(f"AbLang2 residue delta:    {abl_res_delta.shape}")


Loads 6 tensors from Drive. Raw embeddings (mutant sequence-level) are needed only
for the CoV variance ratio comparison. All delta tensors were computed in NB03:
delta = mutant_embedding - wildtype_embedding, aligned row-by-row.

Expected shapes:
- ESM-2 raw sequence: (5318, 2560)
- ESM-2 sequence delta: (5318, 2560)
- ESM-2 residue delta: (5318, 1280)
- AbLang2 raw sequence: (5318, 960)
- AbLang2 sequence delta: (5318, 960)
- AbLang2 residue delta: (5318, 480)


## Sequence-Level EDA

Analyzes the 2560-dim (ESM-2) and 960-dim (AbLang2) sequence-level delta embeddings.
Sequence-level delta = mean_pool(mutant) - mean_pool(wildtype), concatenated over H and L chains.


In [ ]:
# Compute L2 norms of sequence-level delta embeddings
esm2_seq_norms = torch.norm(esm2_seq_delta.float(), dim=1).numpy()
abl_seq_norms  = torch.norm(abl_seq_delta.float(),  dim=1).numpy()

print(f"ESM-2  sequence delta norms: min={esm2_seq_norms.min():.4f}, "
      f"median={np.median(esm2_seq_norms):.4f}, max={esm2_seq_norms.max():.4f}")
print(f"AbLang2 sequence delta norms: min={abl_seq_norms.min():.4f}, "
      f"median={np.median(abl_seq_norms):.4f}, max={abl_seq_norms.max():.4f}")


L2 norm of each row of the delta tensor measures how much the model's representation
changed in response to the mutation. A larger norm = larger perturbation in embedding space.
This is the core quantity used throughout the EDA.

Record confirmed values here after running.


### Discriminability Gain: Raw vs Delta CoV

Coefficient of variation (std / mean of L2 norms) measures how spread out the norm
distribution is relative to its center. Raw sequence embeddings cluster tightly around
the wildtype (high cosine similarity, low CoV). Delta embeddings amplify the mutation-
specific signal. Prior ESM-2 finding: delta CoV is 177x-301x higher than raw CoV.


In [ ]:
plot_delta_variance_ratio(esm2_seq_raw, esm2_seq_delta, dms_names, 'ESM-2',   FIGURES_DIR)
plot_delta_variance_ratio(abl_seq_raw,  abl_seq_delta,  dms_names, 'AbLang2', FIGURES_DIR)


Left panel: raw CoV vs delta CoV per dataset on a log scale.
Right panel: ratio (delta CoV / raw CoV) per dataset.

Known ESM-2 result (from prior work): 177x-301x discriminability gain across datasets.
Record confirmed values and AbLang2 comparison here after running.


### Delta Norm Distributions by Dataset

Violin plots of delta norms per dataset. Verifies that the norm distributions differ
meaningfully across antibodies -- if all five distributions are identical, the model
is not encoding dataset-specific structure.


In [ ]:
plot_delta_norm_by_dataset(esm2_seq_delta, dms_names, 'ESM-2',   FIGURES_DIR)
plot_delta_norm_by_dataset(abl_seq_delta,  dms_names, 'AbLang2', FIGURES_DIR)


Each panel is one AbAgym dataset, showing the distribution of delta norms for all
mutations in that dataset. The five antibodies differ in CDR/FR mutation balance,
so datasets with more CDR mutations (Ang2, VEGF, HER2) may show different norm
distributions than lysozyme (66% FR).

Record observations here after running.


### CDR vs FR Delta Norms (the Inverse CDR Prior)

The CDR prior states that CDR mutations should have larger functional effects than
framework mutations -- CDRs are the binding interface, FR is structural scaffold.
ESM-2 encodes the INVERSE: FR mutations produce larger delta norms than CDR mutations
(Mann-Whitney p=2.78e-122). This is because ESM-2 is trained on general proteins where
FR positions are highly conserved -- any mutation there is unusual and produces a large
representation shift.

Key open question for AbLang2: does antibody-specific pretraining on OAS produce a
different structure? AbLang2 has seen far more CDR diversity than ESM-2, so it may
not be as surprised by CDR mutations.


In [ ]:
plot_delta_norm_cdr_vs_fr(esm2_seq_delta, regions, 'ESM-2',   FIGURES_DIR)
plot_delta_norm_cdr_vs_fr(abl_seq_delta,  regions, 'AbLang2', FIGURES_DIR)


Left panel: CDR vs FR violin with Mann-Whitney annotation and median values.
Right panel: all 7 region types to show CDR loop-level variation.

ESM-2 expected result: FR > CDR, p=2.78e-122, medians ~0.108 vs ~0.084.

For AbLang2: record the direction (CDR > FR or FR > CDR), the Mann-Whitney p-value,
and the median values here after running. Note whether the pattern is reversed,
attenuated, or absent compared to ESM-2.


### Spearman(delta norm, DMS score) -- Sequence Level

Measures how well the L2 norm of the delta embedding correlates with the experimental
mutation effect score (MinMax-normalized DMS). A positive Spearman r means larger
embedding perturbations correspond to larger functional effects.

Prior ESM-2 result (from NB02.5): r = 0.079 to 0.151 per dataset. This is modest but
consistent, and is the baseline signal available without any trained model on top.


In [ ]:
print(f"{'Model':<10} {'Dataset':<35} {'r':>7} {'p':>10}")
print('-' * 65)

seq_spearman = {}
for ds in datasets:
    mask = dms_arr == ds
    for model, norms in [('ESM-2', esm2_seq_norms), ('AbLang2', abl_seq_norms)]:
        r, p = spearmanr(norms[mask], dms_scores[mask])
        seq_spearman[(model, ds)] = r
        print(f"{model:<10} {ds:<35} {r:>7.4f} {p:>10.2e}")
    print()

# Aggregate (all 5318 rows)
for model, norms in [('ESM-2', esm2_seq_norms), ('AbLang2', abl_seq_norms)]:
    r, p = spearmanr(norms, dms_scores)
    seq_spearman[(model, 'ALL')] = r
    print(f"{model:<10} {'ALL':<35} {r:>7.4f} {p:>10.2e}")


Per-dataset Spearman correlation between delta norm and DMS score for both models.

Known ESM-2 sequence-level results (from prior work): r = 0.079 to 0.151 per dataset.

Record confirmed values and AbLang2 results here after running.
Note whether AbLang2 shows stronger or weaker correlation than ESM-2, and whether
HER2 is an outlier (expected: weakest signal, bimodal DMS distribution, all CDR H3).


### PCA Structure -- Sequence Level

PCA of delta embeddings reveals geometric structure in the high-dimensional space.

Known ESM-2 finding: PC1 and PC2 form an orthogonal cross, where PC1 separates heavy
chain mutations and PC2 separates light chain mutations. This cross arises because
ESM-2 embeds H and L in separate forward passes -- the delta affects only one half
of the 2560-dim concatenated vector, producing orthogonal perturbation directions.

AbLang2 hypothesis: cross-chain attention mixes the H/L representations in a single
forward pass, so the orthogonal cross structure may not appear. Whether the H/L
separation is visible in a different form (or absent) is an open question.


In [ ]:
# PCA colored by chain (H vs L) -- tests for orthogonal H/L subspace structure
plot_delta_pca(esm2_seq_delta, np.array(chains), 'chain', 'ESM-2',   FIGURES_DIR, 'sequence')
plot_delta_pca(abl_seq_delta,  np.array(chains), 'chain', 'AbLang2', FIGURES_DIR, 'sequence')


PCA scatter colored by chain identity (H=blue, L=orange).

ESM-2 expected: orthogonal cross shape. H mutations cluster on one axis, L on another.
PC1+PC2 explain ~26% of variance.

For AbLang2: record the shape (cross / diffuse / other), whether H and L separate,
and the explained variance. Note if the structure is qualitatively different from ESM-2.


In [ ]:
# PCA colored by binary CDR/FR
plot_delta_pca(esm2_seq_delta, np.array(cdr_fr), 'cdr_fr', 'ESM-2',   FIGURES_DIR, 'sequence')
plot_delta_pca(abl_seq_delta,  np.array(cdr_fr), 'cdr_fr', 'AbLang2', FIGURES_DIR, 'sequence')


PCA colored by CDR (green) vs FR (red). Reveals whether CDR and FR mutations
occupy distinct regions of the delta embedding space.

Record observations here after running.


In [ ]:
# PCA colored by dataset -- reveals per-antibody clustering
plot_delta_pca(esm2_seq_delta, np.array(dms_names), 'dataset', 'ESM-2',   FIGURES_DIR, 'sequence')
plot_delta_pca(abl_seq_delta,  np.array(dms_names), 'dataset', 'AbLang2', FIGURES_DIR, 'sequence')


PCA colored by dataset. Strong clustering by dataset would indicate that the delta
embeddings are dominated by antibody identity rather than mutation-level signal.

Record observations here after running.


## Residue-Level EDA

Analyzes the 1280-dim (ESM-2) and 480-dim (AbLang2) residue-level delta embeddings.
Residue-level delta = embedding at mutation site (mutant) - embedding at same site (wildtype).

This is the single-token representation at the exact mutation position, in contrast
to sequence-level which aggregates over the entire chain via mean pooling. Residue-level
deltas are used in Experiments 2, 3, 5, and 6.

Key difference for AbLang2: the residue embedding is extracted from a joint VH|VL
forward pass, so it encodes cross-chain context. ESM-2 residue embeddings come from
a single-chain forward pass (only the mutated chain is embedded).


In [ ]:
esm2_res_norms = torch.norm(esm2_res_delta.float(), dim=1).numpy()
abl_res_norms  = torch.norm(abl_res_delta.float(),  dim=1).numpy()

print(f"ESM-2  residue delta norms:  min={esm2_res_norms.min():.4f}, "
      f"median={np.median(esm2_res_norms):.4f}, max={esm2_res_norms.max():.4f}")
print(f"AbLang2 residue delta norms: min={abl_res_norms.min():.4f}, "
      f"median={np.median(abl_res_norms):.4f}, max={abl_res_norms.max():.4f}")


L2 norms of residue-level delta embeddings. Compare to sequence-level norms: residue
deltas should be larger in absolute terms (no dilution from mean pooling over the full
chain), but may have lower CoV if the per-residue representation is less discriminable.

Record confirmed values here after running.


### CDR vs FR Delta Norms -- Residue Level

Tests whether the inverse CDR prior observed at sequence level is also present at
residue level. At the residue level the question is more localized: does the single-
token perturbation at the mutation site differ between CDR and FR positions?


In [ ]:
# Residue-level CDR vs FR -- save with a suffix to distinguish from sequence-level figures
from src.visualization.plots import plot_delta_norm_cdr_vs_fr as _plot_cdr_fr

# We call the same function; filenames are distinguished by model_name
# To separate sequence vs residue files, temporarily patch output names
# by saving to a subdirectory
res_fig_dir = FIGURES_DIR / 'residue_level'
res_fig_dir.mkdir(parents=True, exist_ok=True)

plot_delta_norm_cdr_vs_fr(esm2_res_delta, regions, 'ESM-2',   res_fig_dir)
plot_delta_norm_cdr_vs_fr(abl_res_delta,  regions, 'AbLang2', res_fig_dir)


Residue-level CDR vs FR comparison. Saved to `figures/residue_level/` to distinguish
from the sequence-level versions.

Record: does the same inverse CDR prior appear at residue level? Is it stronger,
weaker, or reversed compared to sequence level?


### Spearman(delta norm, DMS score) -- Residue Level

Residue-level Spearman tells us whether the single-position perturbation norm is a
better or worse predictor of functional effect than the sequence-level norm.
This directly informs whether residue-level embeddings (Experiments 2, 3, 5, 6)
carry useful signal before any training.


In [ ]:
print(f"{'Model':<10} {'Dataset':<35} {'r':>7} {'p':>10}")
print('-' * 65)

res_spearman = {}
for ds in datasets:
    mask = dms_arr == ds
    for model, norms in [('ESM-2', esm2_res_norms), ('AbLang2', abl_res_norms)]:
        r, p = spearmanr(norms[mask], dms_scores[mask])
        res_spearman[(model, ds)] = r
        print(f"{model:<10} {ds:<35} {r:>7.4f} {p:>10.2e}")
    print()

for model, norms in [('ESM-2', esm2_res_norms), ('AbLang2', abl_res_norms)]:
    r, p = spearmanr(norms, dms_scores)
    res_spearman[(model, 'ALL')] = r
    print(f"{model:<10} {'ALL':<35} {r:>7.4f} {p:>10.2e}")


Residue-level Spearman per dataset for both models.

Record confirmed values here after running. Compare to sequence-level r values:
if residue-level r is consistently lower, sequence-level delta is a better unsupervised
signal and the MLP will need to work harder to extract signal from residue-level inputs.


### PCA Structure -- Residue Level

Lower-dimensional residue delta embeddings (1280 and 480 dims). The cross shape
observed in ESM-2 sequence-level PCA may not appear here since only the mutated
chain is embedded per mutation -- H mutations produce 1280-dim ESM-2 residue deltas
with no L contribution, and vice versa.


In [ ]:
plot_delta_pca(esm2_res_delta, np.array(chains), 'chain', 'ESM-2',   res_fig_dir, 'residue')
plot_delta_pca(abl_res_delta,  np.array(chains), 'chain', 'AbLang2', res_fig_dir, 'residue')

plot_delta_pca(esm2_res_delta, np.array(cdr_fr), 'cdr_fr', 'ESM-2',   res_fig_dir, 'residue')
plot_delta_pca(abl_res_delta,  np.array(cdr_fr), 'cdr_fr', 'AbLang2', res_fig_dir, 'residue')


Residue-level PCA colored by chain and by CDR/FR.

For ESM-2 residue: the cross may be absent since H and L mutations are not mixed in
the same forward pass -- each row of the residue delta tensor reflects only one chain.

For AbLang2 residue: the joint forward pass means H and L mutations both have cross-
chain context encoded, so the chain separation may be weaker.

Record observations here after running.


### Sequence vs Residue: Spearman Comparison

Direct side-by-side comparison of sequence-level and residue-level Spearman values
for both models. Informs the expected difficulty of Experiments 2-6 before training.


In [ ]:
print(f"{'Dataset':<35} {'ESM2-Seq':>10} {'ESM2-Res':>10} {'ABL-Seq':>10} {'ABL-Res':>10}")
print('-' * 75)

for ds in datasets + ['ALL']:
    print(
        f"{ds:<35}"
        f"{seq_spearman.get(('ESM-2', ds), float('nan')):>10.4f}"
        f"{res_spearman.get(('ESM-2', ds), float('nan')):>10.4f}"
        f"{seq_spearman.get(('AbLang2', ds), float('nan')):>10.4f}"
        f"{res_spearman.get(('AbLang2', ds), float('nan')):>10.4f}"
    )


Summary comparison table: both models x both embedding levels x 5 datasets + aggregate.

Record the full table here after running. This table will be reproduced in PROGRESS.md
and cited in the paper's supplementary material.

Key question: is one level (sequence vs residue) consistently stronger across both
models and datasets, or does the pattern depend on the antibody?


## Cross-Model Comparison

ESM-2 and AbLang2 are trained on different data with different architectures. Their
delta embeddings may encode the same mutation-level information (high Spearman between
norm vectors), or complementary information (low Spearman, suggesting their outputs
could be combined for better prediction).


In [ ]:
r_seq, p_seq = spearmanr(esm2_seq_norms, abl_seq_norms)
r_res, p_res = spearmanr(esm2_res_norms, abl_res_norms)

print(f"Spearman(ESM-2 seq norms, AbLang2 seq norms): r={r_seq:.4f}, p={p_seq:.2e}")
print(f"Spearman(ESM-2 res norms, AbLang2 res norms): r={r_res:.4f}, p={p_res:.2e}")


Aggregate Spearman between ESM-2 and AbLang2 delta norm vectors. High r (>0.5) means
the models largely agree on which mutations are large vs small. Low r (<0.3) suggests
the models encode different aspects of mutation effect and may benefit from combination.

Record confirmed values here after running.


In [ ]:
plot_model_comparison_norms(esm2_seq_norms, abl_seq_norms, regions, 'sequence', FIGURES_DIR)
plot_model_comparison_norms(esm2_res_norms, abl_res_norms, regions, 'residue',  FIGURES_DIR)


Scatter plots of ESM-2 vs AbLang2 delta norms, colored by CDR (green) vs FR (red).
The dashed diagonal is y=x -- points above it indicate AbLang2 produces a larger norm
for that mutation, and vice versa.

Record observations here after running: do CDR and FR mutations cluster differently
in this space? Is the agreement stronger for one group than the other?


## Summary

Full Spearman table (both models x both levels x all datasets) and key findings.


In [ ]:
rows = []
for ds in datasets + ['ALL']:
    rows.append({
        'Dataset':     ds,
        'ESM-2 Seq':   seq_spearman.get(('ESM-2',   ds), float('nan')),
        'ESM-2 Res':   res_spearman.get(('ESM-2',   ds), float('nan')),
        'AbLang2 Seq': seq_spearman.get(('AbLang2', ds), float('nan')),
        'AbLang2 Res': res_spearman.get(('AbLang2', ds), float('nan')),
    })

summary_df = pd.DataFrame(rows).set_index('Dataset')
print(summary_df.to_string(float_format='{:.4f}'.format))


Full summary table. This is the key output of NB04 and should be copied to PROGRESS.md.

Columns: ESM-2 sequence-level Spearman | ESM-2 residue-level Spearman |
AbLang2 sequence-level Spearman | AbLang2 residue-level Spearman.

Record the full table here after running.
